# SentinelFlow AI — Model Training

## Objective

This notebook develops and evaluates the first machine-learning models for SentinelFlow AI.

The initial focus is Stage 1:

**Binary intrusion detection**

- `0` = BENIGN
- `1` = ATTACK

The goal is to train models that can distinguish normal network traffic from malicious traffic using the 65 features selected during the feature-engineering stage.

## Development Strategy

- Rebuild the final clean ML dataset
- Reproduce the same 70/15/15 train-validation-test split
- Keep the test set untouched during model development
- Train baseline models
- Compare models using appropriate intrusion-detection metrics
- Select the best model for final evaluation

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn

PROCESSED_DIR = Path("data/processed")
processed_files = sorted(PROCESSED_DIR.glob("*.csv"))

RANDOM_STATE = 42

print(f"Processed files found: {len(processed_files)}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Random state: {RANDOM_STATE}")

Processed files found: 8
scikit-learn version: 1.9.0
Random state: 42


In [3]:
selected_features = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Fwd Packet Length Max",
    "Fwd Packet Length Min",
    "Fwd Packet Length Mean",
    "Fwd Packet Length Std",
    "Bwd Packet Length Max",
    "Bwd Packet Length Min",
    "Bwd Packet Length Mean",
    "Bwd Packet Length Std",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Total",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Max",
    "Fwd IAT Min",
    "Bwd IAT Total",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",
    "Fwd PSH Flags",
    "Fwd URG Flags",
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Min Packet Length",
    "Max Packet Length",
    "Packet Length Mean",
    "Packet Length Std",
    "Packet Length Variance",
    "FIN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "URG Flag Count",
    "ECE Flag Count",
    "Down/Up Ratio",
    "Average Packet Size",
    "Avg Fwd Segment Size",
    "Avg Bwd Segment Size",
    "Subflow Fwd Bytes",
    "Subflow Bwd Bytes",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
    "act_data_pkt_fwd",
    "min_seg_size_forward",
    "Active Mean",
    "Active Std",
    "Active Max",
    "Active Min",
    "Idle Mean",
    "Idle Std",
    "Idle Max",
    "Idle Min"
]

print(f"Selected features: {len(selected_features)}")

Selected features: 65


In [4]:
# Rebuild the clean ML dataset

ml_parts = []

for file in processed_files:
    print(f"Loading: {file.name}")

    part = pd.read_csv(
        file,
        usecols=selected_features + ["Label"],
        low_memory=False
    )

    ml_parts.append(part)

ml_df = pd.concat(
    ml_parts,
    ignore_index=True
)

original_rows = len(ml_df)

# Remove exact duplicate rows
ml_df = ml_df.drop_duplicates(
    subset=selected_features + ["Label"],
    keep="first"
).reset_index(drop=True)

# Identify conflicting feature vectors
feature_hash = pd.util.hash_pandas_object(
    ml_df[selected_features],
    index=False
)

ml_df["_feature_hash"] = feature_hash

labels_per_features = (
    ml_df.groupby("_feature_hash")["Label"]
    .nunique()
)

conflicting_hashes = labels_per_features[
    labels_per_features > 1
].index

# Remove conflicting groups
ml_df = ml_df[
    ~ml_df["_feature_hash"].isin(conflicting_hashes)
].copy()

ml_df.drop(columns="_feature_hash", inplace=True)
ml_df.reset_index(drop=True, inplace=True)

# Create binary target
ml_df["binary_target"] = (
    ml_df["Label"] != "BENIGN"
).astype("int8")

print("\n--- Clean ML Dataset ---")
print(f"Original rows: {original_rows:,}")
print(f"Final rows: {len(ml_df):,}")
print(f"Selected features: {len(selected_features)}")
print(
    f"BENIGN: {(ml_df['binary_target'] == 0).sum():,}"
)
print(
    f"ATTACK: {(ml_df['binary_target'] == 1).sum():,}"
)

Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Wednesday-workingHours.pcap_ISCX.csv

--- Clean ML Dataset ---
Original rows: 2,572,640
Final rows: 2,519,404
Selected features: 65
BENIGN: 2,094,360
ATTACK: 425,044


In [7]:
from sklearn.model_selection import train_test_split

all_indices = np.arange(len(ml_df))

train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=ml_df["Label"]
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=ml_df.iloc[temp_idx]["Label"]
)

print("--- Dataset Split ---")
print(f"Train:      {len(train_idx):,}")
print(f"Validation: {len(val_idx):,}")
print(f"Test:       {len(test_idx):,}")

--- Dataset Split ---
Train:      1,763,582
Validation: 377,911
Test:       377,911


In [8]:
# Prepare training and validation data for Stage 1
# Binary intrusion detection: BENIGN vs ATTACK

X_train = ml_df.iloc[train_idx][selected_features]
y_train = ml_df.iloc[train_idx]["binary_target"]

X_val = ml_df.iloc[val_idx][selected_features]
y_val = ml_df.iloc[val_idx]["binary_target"]

print("--- Stage 1 Data ---")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

print(f"\nX_validation: {X_val.shape}")
print(f"y_validation: {y_val.shape}")

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

--- Stage 1 Data ---
X_train: (1763582, 65)
y_train: (1763582,)

X_validation: (377911, 65)
y_validation: (377911,)

Training target distribution:
binary_target
0    1466051
1     297531
Name: count, dtype: int64


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=500,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

print("Training Logistic Regression baseline...")

logreg_model.fit(
    X_train,
    y_train
)

print("Training complete.")

Training Logistic Regression baseline...


C:\Users\abdal\SentinelFlow-AI\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Training complete.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Predict on validation data
y_val_pred = logreg_model.predict(X_val)

accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

print("--- Logistic Regression Validation Results ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

In [ ]:
small_pred = logreg_model.predict(X_val.iloc[:1000])

print("Predictions completed.")
print(small_pred[:20])

In [9]:
from sklearn.model_selection import train_test_split

# Create a representative training sample
X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train,
    y_train,
    train_size=300000,
    random_state=RANDOM_STATE,
    stratify=y_train
)

print("Training sample shape:", X_train_sample.shape)
print("\nTarget distribution:")
print(y_train_sample.value_counts().sort_index())

Training sample shape: (300000, 65)

Target distribution:
binary_target
0    249387
1     50613
Name: count, dtype: int64


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=300,
            random_state=RANDOM_STATE
        )
    )
])

print("Training Logistic Regression on 300,000 rows...")

logreg_model.fit(
    X_train_sample,
    y_train_sample
)

print("Training complete.")

Training Logistic Regression on 300,000 rows...
Training complete.


In [11]:
# Quick validation test on 10,000 rows

X_val_small = X_val.iloc[:10000]
y_val_small = y_val.iloc[:10000]

y_val_small_pred = logreg_model.predict(X_val_small)

print("Prediction complete.")
print(f"Predictions: {len(y_val_small_pred):,}")

Prediction complete.
Predictions: 10,000


In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

accuracy = accuracy_score(y_val_small, y_val_small_pred)
precision = precision_score(y_val_small, y_val_small_pred)
recall = recall_score(y_val_small, y_val_small_pred)
f1 = f1_score(y_val_small, y_val_small_pred)

print("--- Logistic Regression Quick Validation ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val_small, y_val_small_pred))

--- Logistic Regression Quick Validation ---
Accuracy:  0.9413
Precision: 0.7596
Recall:    0.9768
F1 Score:  0.8546

Confusion Matrix:
[[7688  546]
 [  41 1725]]


In [13]:
validation_predictions = []

batch_size = 10000

print("Starting validation prediction...")

for start in range(0, len(X_val), batch_size):
    end = min(start + batch_size, len(X_val))

    batch_predictions = logreg_model.predict(
        X_val.iloc[start:end]
    )

    validation_predictions.append(batch_predictions)

    if end % 50000 == 0 or end == len(X_val):
        print(f"Processed {end:,} / {len(X_val):,}")

y_val_pred = np.concatenate(validation_predictions)

print("\nValidation prediction complete.")
print(f"Total predictions: {len(y_val_pred):,}")

Starting validation prediction...
Processed 50,000 / 377,911
Processed 100,000 / 377,911
Processed 150,000 / 377,911
Processed 200,000 / 377,911
Processed 250,000 / 377,911
Processed 300,000 / 377,911
Processed 350,000 / 377,911
Processed 377,911 / 377,911

Validation prediction complete.
Total predictions: 377,911


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

cm = confusion_matrix(y_val, y_val_pred)

print("--- Logistic Regression Full Validation Results ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\n--- Detection Summary ---")
print(f"True Negatives:  {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives:  {tp:,}")

--- Logistic Regression Full Validation Results ---
Accuracy:  0.9377
Precision: 0.7396
Recall:    0.9735
F1 Score:  0.8406

Confusion Matrix:
[[292297  21858]
 [  1689  62067]]

--- Detection Summary ---
True Negatives:  292,297
False Positives: 21,858
False Negatives: 1,689
True Positives:  62,067


## Baseline Model — Logistic Regression

A Logistic Regression baseline was trained using a stratified sample of 300,000 training flows.

The model was evaluated on the complete validation set of 377,911 flows.

### Validation Results

- Accuracy: 0.9377
- Precision: 0.7396
- Recall: 0.9735
- F1 Score: 0.8406
- True Negatives: 292,297
- False Positives: 21,858
- False Negatives: 1,689
- True Positives: 62,067

### Interpretation

The baseline achieves very high attack recall, detecting 97.35% of malicious flows in the validation set.

However, the model produces a relatively high number of false-positive alerts. The next modeling stage will therefore focus on improving precision and reducing false positives while maintaining strong attack recall.

The test set remains untouched during model development.

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training Random Forest on 300,000 rows...")

rf_model.fit(
    X_train_sample,
    y_train_sample
)

print("Training complete.")

Training Random Forest on 300,000 rows...
Training complete.


In [16]:
print(type(rf_model))
print(f"Number of trees: {len(rf_model.estimators_)}")

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
Number of trees: 100


In [17]:
rf_validation_predictions = []

batch_size = 10000

print("Starting Random Forest validation prediction...")

for start in range(0, len(X_val), batch_size):
    end = min(start + batch_size, len(X_val))

    batch_predictions = rf_model.predict(
        X_val.iloc[start:end]
    )

    rf_validation_predictions.append(batch_predictions)

    if end % 50000 == 0 or end == len(X_val):
        print(f"Processed {end:,} / {len(X_val):,}")

rf_val_pred = np.concatenate(rf_validation_predictions)

print("\nRandom Forest validation prediction complete.")
print(f"Total predictions: {len(rf_val_pred):,}")

Starting Random Forest validation prediction...
Processed 50,000 / 377,911
Processed 100,000 / 377,911
Processed 150,000 / 377,911
Processed 200,000 / 377,911
Processed 250,000 / 377,911
Processed 300,000 / 377,911
Processed 350,000 / 377,911
Processed 377,911 / 377,911

Random Forest validation prediction complete.
Total predictions: 377,911


In [18]:
rf_accuracy = accuracy_score(y_val, rf_val_pred)
rf_precision = precision_score(y_val, rf_val_pred)
rf_recall = recall_score(y_val, rf_val_pred)
rf_f1 = f1_score(y_val, rf_val_pred)

rf_cm = confusion_matrix(y_val, rf_val_pred)

print("--- Random Forest Full Validation Results ---")
print(f"Accuracy:  {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1 Score:  {rf_f1:.4f}")

print("\nConfusion Matrix:")
print(rf_cm)

rf_tn, rf_fp, rf_fn, rf_tp = rf_cm.ravel()

print("\n--- Detection Summary ---")
print(f"True Negatives:  {rf_tn:,}")
print(f"False Positives: {rf_fp:,}")
print(f"False Negatives: {rf_fn:,}")
print(f"True Positives:  {rf_tp:,}")

--- Random Forest Full Validation Results ---
Accuracy:  0.9990
Precision: 0.9962
Recall:    0.9979
F1 Score:  0.9970

Confusion Matrix:
[[313912    243]
 [   134  63622]]

--- Detection Summary ---
True Negatives:  313,912
False Positives: 243
False Negatives: 134
True Positives:  63,622


## Random Forest — Validation Results

A Random Forest classifier with 100 trees was trained on the same 300,000-flow training sample used for the Logistic Regression baseline.

### Validation Results

- Accuracy: 0.9990
- Precision: 0.9962
- Recall: 0.9979
- F1 Score: 0.9970
- True Negatives: 313,912
- False Positives: 243
- False Negatives: 134
- True Positives: 63,622

### Comparison with Logistic Regression

Random Forest substantially outperformed the Logistic Regression baseline.

It reduced false positives from 21,858 to 243 and false negatives from 1,689 to 134 while achieving an F1 score of 0.9970.

Because this performance is extremely high, additional validation is required before final model selection. In particular, the CIC-IDS2017 dataset contains strong source/day and attack-pattern structure, so random stratified validation may produce optimistic estimates of real-world generalization.

The test set remains untouched.

In [19]:
# Analyze false negatives by original attack type

val_results = ml_df.iloc[val_idx][["Label"]].copy()
val_results["binary_target"] = y_val.to_numpy()
val_results["rf_prediction"] = rf_val_pred

false_negatives = val_results[
    (val_results["binary_target"] == 1) &
    (val_results["rf_prediction"] == 0)
]

fn_by_attack = (
    false_negatives["Label"]
    .value_counts()
    .sort_values(ascending=False)
)

print("--- Random Forest False Negatives by Attack Type ---")
print(f"Total false negatives: {len(false_negatives):,}\n")
print(fn_by_attack)

--- Random Forest False Negatives by Attack Type ---
Total false negatives: 134

Label
Bot                         49
DoS Hulk                    33
DDoS                         9
DoS Slowhttptest             7
DoS GoldenEye                7
DoS slowloris                5
Web Attack Brute Force       5
Infiltration                 5
PortScan                     4
SSH-Patator                  3
FTP-Patator                  3
Web Attack Sql Injection     2
Heartbleed                   1
Web Attack XSS               1
Name: count, dtype: int64


In [20]:
# Recall for each attack type

attack_validation = val_results[
    val_results["binary_target"] == 1
].copy()

attack_performance = (
    attack_validation
    .groupby("Label")
    .agg(
        Total=("Label", "size"),
        Detected=("rf_prediction", "sum")
    )
)

attack_performance["Missed"] = (
    attack_performance["Total"] -
    attack_performance["Detected"]
)

attack_performance["Recall"] = (
    attack_performance["Detected"] /
    attack_performance["Total"]
)

attack_performance = attack_performance.sort_values(
    "Recall"
)

print("--- Random Forest Recall by Attack Type ---")
print(attack_performance)

--- Random Forest Recall by Attack Type ---
                          Total  Detected  Missed    Recall
Label                                                      
Infiltration                  5         0       5  0.000000
Web Attack Sql Injection      3         1       2  0.333333
Heartbleed                    2         1       1  0.500000
Bot                         292       243      49  0.832192
Web Attack Brute Force      221       216       5  0.977376
Web Attack XSS               98        97       1  0.989796
DoS Slowhttptest            784       777       7  0.991071
SSH-Patator                 483       480       3  0.993789
DoS slowloris               807       802       5  0.993804
DoS GoldenEye              1543      1536       7  0.995463
FTP-Patator                 889       886       3  0.996625
DoS Hulk                  25907     25874      33  0.998726
DDoS                      19202     19193       9  0.999531
PortScan                  13520     13516       4  0.999

### Attack-Level Error Analysis

Although the Random Forest achieved excellent overall binary-detection performance, attack-level analysis revealed important limitations.

The model performed extremely well on high-frequency attack classes such as:

- PortScan: 99.97% recall
- DDoS: 99.95% recall
- DoS Hulk: 99.87% recall
- FTP-Patator: 99.66% recall

However, performance was substantially weaker for several rare attack classes:

- Bot: 83.22% recall (243/292 detected)
- Heartbleed: 50.00% recall (1/2 detected)
- Web Attack SQL Injection: 33.33% recall (1/3 detected)
- Infiltration: 0.00% recall (0/5 detected)

These results demonstrate why aggregate metrics alone are insufficient for evaluating an intrusion-detection system.

The very small validation counts for rare attacks also make their individual recall estimates unstable. Therefore, these results should be interpreted cautiously rather than as reliable estimates of real-world performance.

Further evaluation and training strategies should specifically consider rare attack classes.

In [21]:
# Build a 300,000-row training sample stratified by the original 15 classes

train_labels_multiclass = ml_df.iloc[train_idx]["Label"]

sample_indices, _ = train_test_split(
    np.arange(len(X_train)),
    train_size=300000,
    random_state=RANDOM_STATE,
    stratify=train_labels_multiclass
)

X_train_stratified = X_train.iloc[sample_indices]
y_train_stratified = y_train.iloc[sample_indices]

sample_labels = train_labels_multiclass.iloc[sample_indices]

print(f"Training sample shape: {X_train_stratified.shape}")
print(f"Number of attack classes represented: {sample_labels.nunique()}")
print("\nClass distribution:")
print(sample_labels.value_counts())

Training sample shape: (300000, 65)
Number of attack classes represented: 15

Class distribution:
Label
BENIGN                      249388
DoS Hulk                     20567
DDoS                         15243
PortScan                     10732
DoS GoldenEye                 1225
FTP-Patator                    706
DoS slowloris                  641
DoS Slowhttptest               622
SSH-Patator                    383
Bot                            232
Web Attack Brute Force         175
Web Attack XSS                  78
Infiltration                     4
Web Attack Sql Injection         3
Heartbleed                       1
Name: count, dtype: int64


In [22]:
# Check full training-set class distribution

full_train_labels = ml_df.iloc[train_idx]["Label"]

print("--- Full Training Set Class Distribution ---")
print(full_train_labels.value_counts())

print("\n--- Rare Attack Classes ---")

rare_attacks = [
    "Infiltration",
    "Web Attack Sql Injection",
    "Heartbleed"
]

for attack in rare_attacks:
    count = (full_train_labels == attack).sum()
    print(f"{attack}: {count}")

--- Full Training Set Class Distribution ---
Label
BENIGN                      1466051
DoS Hulk                     120902
DDoS                          89608
PortScan                      63091
DoS GoldenEye                  7200
FTP-Patator                    4152
DoS slowloris                  3769
DoS Slowhttptest               3659
SSH-Patator                    2253
Bot                            1364
Web Attack Brute Force         1029
Web Attack XSS                  456
Infiltration                     25
Web Attack Sql Injection         15
Heartbleed                        8
Name: count, dtype: int64

--- Rare Attack Classes ---
Infiltration: 25
Web Attack Sql Injection: 15
Heartbleed: 8


In [23]:
from sklearn.ensemble import RandomForestClassifier

rf_full_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training Random Forest on the full training set...")
print(f"Training rows: {len(X_train):,}")

rf_full_model.fit(X_train, y_train)

print("Full training complete.")

Training Random Forest on the full training set...
Training rows: 1,763,582
Full training complete.


In [24]:
rf_full_validation_predictions = []

batch_size = 10000

print("Starting full-training Random Forest validation prediction...")

for start in range(0, len(X_val), batch_size):
    end = min(start + batch_size, len(X_val))

    batch_predictions = rf_full_model.predict(
        X_val.iloc[start:end]
    )

    rf_full_validation_predictions.append(batch_predictions)

    if end % 50000 == 0 or end == len(X_val):
        print(f"Processed {end:,} / {len(X_val):,}")

rf_full_val_pred = np.concatenate(
    rf_full_validation_predictions
)

print("\nValidation prediction complete.")
print(f"Total predictions: {len(rf_full_val_pred):,}")

Starting full-training Random Forest validation prediction...
Processed 50,000 / 377,911
Processed 100,000 / 377,911
Processed 150,000 / 377,911
Processed 200,000 / 377,911
Processed 250,000 / 377,911
Processed 300,000 / 377,911
Processed 350,000 / 377,911
Processed 377,911 / 377,911

Validation prediction complete.
Total predictions: 377,911


In [25]:
rf_full_accuracy = accuracy_score(y_val, rf_full_val_pred)
rf_full_precision = precision_score(y_val, rf_full_val_pred)
rf_full_recall = recall_score(y_val, rf_full_val_pred)
rf_full_f1 = f1_score(y_val, rf_full_val_pred)

rf_full_cm = confusion_matrix(y_val, rf_full_val_pred)
rf_full_tn, rf_full_fp, rf_full_fn, rf_full_tp = rf_full_cm.ravel()

print("--- Full-Training Random Forest Validation Results ---")
print(f"Accuracy:  {rf_full_accuracy:.4f}")
print(f"Precision: {rf_full_precision:.4f}")
print(f"Recall:    {rf_full_recall:.4f}")
print(f"F1 Score:  {rf_full_f1:.4f}")

print("\nConfusion Matrix:")
print(rf_full_cm)

print("\n--- Detection Summary ---")
print(f"True Negatives:  {rf_full_tn:,}")
print(f"False Positives: {rf_full_fp:,}")
print(f"False Negatives: {rf_full_fn:,}")
print(f"True Positives:  {rf_full_tp:,}")

print("\n--- Comparison with 300K Random Forest ---")
print(f"F1:              0.9970 -> {rf_full_f1:.4f}")
print(f"False Positives: 243 -> {rf_full_fp:,}")
print(f"False Negatives: 134 -> {rf_full_fn:,}")

--- Full-Training Random Forest Validation Results ---
Accuracy:  0.9992
Precision: 0.9961
Recall:    0.9992
F1 Score:  0.9977

Confusion Matrix:
[[313907    248]
 [    52  63704]]

--- Detection Summary ---
True Negatives:  313,907
False Positives: 248
False Negatives: 52
True Positives:  63,704

--- Comparison with 300K Random Forest ---
F1:              0.9970 -> 0.9977
False Positives: 243 -> 248
False Negatives: 134 -> 52


In [26]:
# Analyze full-training Random Forest performance by attack type

full_val_results = ml_df.iloc[val_idx][["Label"]].copy()
full_val_results["binary_target"] = y_val.to_numpy()
full_val_results["rf_prediction"] = rf_full_val_pred

attack_validation_full = full_val_results[
    full_val_results["binary_target"] == 1
].copy()

attack_performance_full = (
    attack_validation_full
    .groupby("Label")
    .agg(
        Total=("Label", "size"),
        Detected=("rf_prediction", "sum")
    )
)

attack_performance_full["Missed"] = (
    attack_performance_full["Total"]
    - attack_performance_full["Detected"]
)

attack_performance_full["Recall"] = (
    attack_performance_full["Detected"]
    / attack_performance_full["Total"]
)

attack_performance_full = attack_performance_full.sort_values("Recall")

print("--- Full-Training Random Forest Recall by Attack Type ---")
print(attack_performance_full)

print(
    "\nTotal missed attacks:",
    attack_performance_full["Missed"].sum()
)

--- Full-Training Random Forest Recall by Attack Type ---
                          Total  Detected  Missed    Recall
Label                                                      
Web Attack Sql Injection      3         2       1  0.666667
Infiltration                  5         4       1  0.800000
Bot                         292       271      21  0.928082
Web Attack XSS               98        97       1  0.989796
Web Attack Brute Force      221       219       2  0.990950
DoS GoldenEye              1543      1538       5  0.996760
DoS Slowhttptest            784       782       2  0.997449
DoS slowloris               807       805       2  0.997522
SSH-Patator                 483       482       1  0.997930
DoS Hulk                  25907     25896      11  0.999575
DDoS                      19202     19199       3  0.999844
PortScan                  13520     13518       2  0.999852
Heartbleed                    2         2       0  1.000000
FTP-Patator                 889       889 

### Full-Training Random Forest — Attack-Level Analysis

Training the Random Forest on the complete training set of 1,763,582 flows substantially improved attack detection compared with the 300,000-flow model.

Overall validation performance:

- Accuracy: 99.92%
- Precision: 99.61%
- Recall: 99.92%
- F1 Score: 99.77%
- False Positives: 248
- False Negatives: 52

The number of missed attacks decreased from 134 to 52.

Notable improvements among previously difficult attack types included:

- Infiltration: 0% → 80.00% recall
- Web Attack SQL Injection: 33.33% → 66.67% recall
- Bot: 83.22% → 92.81% recall
- Heartbleed: 50.00% → 100.00% recall

Several high-frequency attacks achieved recall above 99.9%, including DDoS and PortScan.

Despite these strong results, rare-class metrics must be interpreted cautiously. For example, the validation set contains only 2 Heartbleed, 3 SQL Injection, and 5 Infiltration flows. Therefore, their percentages are not reliable estimates of real-world generalization.

The test set remains untouched.

In [27]:
# Inspect processed source files before source/day stress testing

print("--- Processed Source Files ---\n")

for file in processed_files:
    row_count = sum(1 for _ in open(file, encoding="utf-8", errors="ignore")) - 1
    print(f"{file.name}: {row_count:,} rows")

--- Processed Source Files ---

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 223,082 rows
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 213,777 rows
Friday-WorkingHours-Morning.pcap_ISCX.csv: 184,044 rows
Monday-WorkingHours.pcap_ISCX.csv: 502,650 rows
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 252,790 rows
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 164,179 rows
Tuesday-WorkingHours.pcap_ISCX.csv: 421,626 rows
Wednesday-workingHours.pcap_ISCX.csv: 610,492 rows


In [28]:
# Inspect attack distribution inside each processed source file

for file in processed_files:
    labels = pd.read_csv(
        file,
        usecols=["Label"],
        low_memory=False
    )["Label"].value_counts()

    print(f"\n--- {file.name} ---")
    print(labels)


--- Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv ---
Label
DDoS      128014
BENIGN     95068
Name: count, dtype: int64

--- Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv ---
Label
BENIGN      123083
PortScan     90694
Name: count, dtype: int64

--- Friday-WorkingHours-Morning.pcap_ISCX.csv ---
Label
BENIGN    182096
Bot         1948
Name: count, dtype: int64

--- Monday-WorkingHours.pcap_ISCX.csv ---
Label
BENIGN    502650
Name: count, dtype: int64

--- Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv ---
Label
BENIGN          252754
Infiltration        36
Name: count, dtype: int64

--- Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv ---
Label
BENIGN                      162036
Web Attack Brute Force        1470
Web Attack XSS                 652
Web Attack Sql Injection        21
Name: count, dtype: int64

--- Tuesday-WorkingHours.pcap_ISCX.csv ---
Label
BENIGN         412476
FTP-Patator      5931
SSH-Patator      3219
Name: count, dtype: int64

--- Wedne

In [29]:
# Source/day stress test setup
# Hold out the complete Friday Morning (Bot) source file

STRESS_TEST_FILE = "Friday-WorkingHours-Morning.pcap_ISCX.csv"

stress_train_parts = []
stress_test_df = None

for file in processed_files:
    part = pd.read_csv(
        file,
        usecols=selected_features + ["Label"],
        low_memory=False
    )

    if file.name == STRESS_TEST_FILE:
        stress_test_df = part.copy()
    else:
        stress_train_parts.append(part)

stress_train_df = pd.concat(
    stress_train_parts,
    ignore_index=True
)

print("--- Source-Holdout Stress Test Setup ---")
print(f"Held-out source: {STRESS_TEST_FILE}")
print(f"Training rows before ML cleaning: {len(stress_train_df):,}")
print(f"Held-out rows: {len(stress_test_df):,}")

print("\nHeld-out label distribution:")
print(stress_test_df["Label"].value_counts())

--- Source-Holdout Stress Test Setup ---
Held-out source: Friday-WorkingHours-Morning.pcap_ISCX.csv
Training rows before ML cleaning: 2,388,596
Held-out rows: 184,044

Held-out label distribution:
Label
BENIGN    182096
Bot         1948
Name: count, dtype: int64


In [30]:
# Clean stress-test training data only

stress_original_rows = len(stress_train_df)

# 1. Remove exact duplicate feature + label rows
stress_train_df = stress_train_df.drop_duplicates(
    subset=selected_features + ["Label"],
    keep="first"
).reset_index(drop=True)

stress_after_duplicates = len(stress_train_df)

# 2. Detect feature vectors associated with multiple labels
stress_feature_hash = pd.util.hash_pandas_object(
    stress_train_df[selected_features],
    index=False
)

stress_train_df["_feature_hash"] = stress_feature_hash

stress_labels_per_features = (
    stress_train_df
    .groupby("_feature_hash")["Label"]
    .nunique()
)

stress_conflicting_hashes = stress_labels_per_features[
    stress_labels_per_features > 1
].index

stress_conflicting_rows = stress_train_df[
    stress_train_df["_feature_hash"].isin(stress_conflicting_hashes)
].shape[0]

# 3. Remove complete conflicting groups
stress_train_df = stress_train_df[
    ~stress_train_df["_feature_hash"].isin(stress_conflicting_hashes)
].copy()

stress_train_df.drop(
    columns="_feature_hash",
    inplace=True
)

stress_train_df.reset_index(
    drop=True,
    inplace=True
)

# 4. Create binary target
stress_train_df["binary_target"] = (
    stress_train_df["Label"] != "BENIGN"
).astype("int8")

print("--- Stress-Test Training Cleaning ---")
print(f"Original training rows:       {stress_original_rows:,}")
print(f"Exact duplicates removed:     {stress_original_rows - stress_after_duplicates:,}")
print(f"Conflicting rows removed:     {stress_conflicting_rows:,}")
print(f"Final training rows:          {len(stress_train_df):,}")

print("\nBinary target distribution:")
print(stress_train_df["binary_target"].value_counts())

--- Stress-Test Training Cleaning ---
Original training rows:       2,388,596
Exact duplicates removed:     39,952
Conflicting rows removed:     1,374
Final training rows:          2,347,270

Binary target distribution:
binary_target
0    1924164
1     423106
Name: count, dtype: int64


In [31]:
# Prepare source-holdout training and test data

X_stress_train = stress_train_df[selected_features]
y_stress_train = stress_train_df["binary_target"]

X_stress_test = stress_test_df[selected_features]
y_stress_test = (
    stress_test_df["Label"] != "BENIGN"
).astype("int8")

print("--- Stress-Test Data Ready ---")
print(f"Training shape: {X_stress_train.shape}")
print(f"Held-out test shape: {X_stress_test.shape}")

print("\nTraining Bot examples:")
print((stress_train_df["Label"] == "Bot").sum())

print("\nHeld-out target distribution:")
print(y_stress_test.value_counts())

--- Stress-Test Data Ready ---
Training shape: (2347270, 65)
Held-out test shape: (184044, 65)

Training Bot examples:
0

Held-out target distribution:
Label
0    182096
1      1948
Name: count, dtype: int64


In [32]:
from sklearn.ensemble import RandomForestClassifier

rf_stress_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training source-holdout Random Forest...")
print(f"Training rows: {len(X_stress_train):,}")
print("Bot examples seen during training: 0")

rf_stress_model.fit(
    X_stress_train,
    y_stress_train
)

print("\nStress-test model training complete.")

Training source-holdout Random Forest...
Training rows: 2,347,270
Bot examples seen during training: 0

Stress-test model training complete.


In [33]:
stress_predictions = []

batch_size = 10000

print("Starting source-holdout prediction...")

for start in range(0, len(X_stress_test), batch_size):
    end = min(start + batch_size, len(X_stress_test))

    batch_predictions = rf_stress_model.predict(
        X_stress_test.iloc[start:end]
    )

    stress_predictions.append(batch_predictions)

    if end % 50000 == 0 or end == len(X_stress_test):
        print(
            f"Processed {end:,} / "
            f"{len(X_stress_test):,}"
        )

y_stress_pred = np.concatenate(stress_predictions)

print("\nSource-holdout prediction complete.")
print(f"Total predictions: {len(y_stress_pred):,}")

Starting source-holdout prediction...
Processed 50,000 / 184,044
Processed 100,000 / 184,044
Processed 150,000 / 184,044
Processed 184,044 / 184,044

Source-holdout prediction complete.
Total predictions: 184,044


In [34]:
stress_accuracy = accuracy_score(
    y_stress_test,
    y_stress_pred
)

stress_precision = precision_score(
    y_stress_test,
    y_stress_pred,
    zero_division=0
)

stress_recall = recall_score(
    y_stress_test,
    y_stress_pred,
    zero_division=0
)

stress_f1 = f1_score(
    y_stress_test,
    y_stress_pred,
    zero_division=0
)

stress_cm = confusion_matrix(
    y_stress_test,
    y_stress_pred
)

stress_tn, stress_fp, stress_fn, stress_tp = stress_cm.ravel()

print("--- Source-Holdout Stress Test Results ---")
print(f"Accuracy:  {stress_accuracy:.4f}")
print(f"Precision: {stress_precision:.4f}")
print(f"Recall:    {stress_recall:.4f}")
print(f"F1 Score:  {stress_f1:.4f}")

print("\nConfusion Matrix:")
print(stress_cm)

print("\n--- Detection Summary ---")
print(f"True Negatives:  {stress_tn:,}")
print(f"False Positives: {stress_fp:,}")
print(f"False Negatives: {stress_fn:,}")
print(f"True Positives:  {stress_tp:,}")

print("\n--- Unseen Bot Detection ---")
print(f"Bot flows in held-out source: {stress_tp + stress_fn:,}")
print(f"Bot flows detected:           {stress_tp:,}")
print(f"Bot flows missed:             {stress_fn:,}")
print(f"Bot detection rate:           {stress_recall:.2%}")

--- Source-Holdout Stress Test Results ---
Accuracy:  0.9891
Precision: 0.0000
Recall:    0.0000
F1 Score:  0.0000

Confusion Matrix:
[[182032     64]
 [  1948      0]]

--- Detection Summary ---
True Negatives:  182,032
False Positives: 64
False Negatives: 1,948
True Positives:  0

--- Unseen Bot Detection ---
Bot flows in held-out source: 1,948
Bot flows detected:           0
Bot flows missed:             1,948
Bot detection rate:           0.00%


## Source-Holdout Generalization Stress Test

To evaluate generalization beyond the random stratified split, the complete Friday Morning source file was held out from training.

This source contains:

- 182,096 BENIGN flows
- 1,948 Bot flows

Because Bot traffic occurs only in this held-out source, the model was trained with **zero Bot examples**.

A Random Forest was trained on the remaining 2,347,270 cleaned flows and evaluated on the complete held-out source.

### Results

- Accuracy: 98.91%
- Precision: 0.00%
- Recall: 0.00%
- F1 Score: 0.00%
- True Negatives: 182,032
- False Positives: 64
- False Negatives: 1,948
- True Positives: 0

The model failed to detect any of the 1,948 previously unseen Bot flows.

### Interpretation

The high accuracy is misleading because the held-out source is overwhelmingly BENIGN.

This experiment demonstrates an important limitation of the supervised detector: excellent performance on a random stratified validation split does not imply reliable detection of previously unseen attack patterns.

The Random Forest generalizes very well to new examples of attack classes represented during training, but it did not generalize from known attacks to the unseen Bot attack pattern in this source-holdout experiment.

This result motivates future integration of anomaly-detection techniques alongside supervised classification to improve detection of novel or unseen threats.

In [35]:
# Stage 2 — Multiclass Attack Classification

train_stage2 = ml_df.iloc[train_idx].copy()
val_stage2 = ml_df.iloc[val_idx].copy()

# Keep attacks only
train_stage2 = train_stage2[
    train_stage2["Label"] != "BENIGN"
].copy()

val_stage2 = val_stage2[
    val_stage2["Label"] != "BENIGN"
].copy()

X_train_stage2 = train_stage2[selected_features]
y_train_stage2 = train_stage2["Label"]

X_val_stage2 = val_stage2[selected_features]
y_val_stage2 = val_stage2["Label"]

print("--- Stage 2 Data ---")
print(f"Training attack flows:   {len(X_train_stage2):,}")
print(f"Validation attack flows: {len(X_val_stage2):,}")
print(f"Attack classes:          {y_train_stage2.nunique()}")

print("\nTraining class distribution:")
print(y_train_stage2.value_counts())

print("\nValidation class distribution:")
print(y_val_stage2.value_counts())

--- Stage 2 Data ---
Training attack flows:   297,531
Validation attack flows: 63,756
Attack classes:          14

Training class distribution:
Label
DoS Hulk                    120902
DDoS                         89608
PortScan                     63091
DoS GoldenEye                 7200
FTP-Patator                   4152
DoS slowloris                 3769
DoS Slowhttptest              3659
SSH-Patator                   2253
Bot                           1364
Web Attack Brute Force        1029
Web Attack XSS                 456
Infiltration                    25
Web Attack Sql Injection        15
Heartbleed                       8
Name: count, dtype: int64

Validation class distribution:
Label
DoS Hulk                    25907
DDoS                        19202
PortScan                    13520
DoS GoldenEye                1543
FTP-Patator                   889
DoS slowloris                 807
DoS Slowhttptest              784
SSH-Patator                   483
Bot                     

In [36]:
from sklearn.ensemble import RandomForestClassifier

rf_multiclass_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=25,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training Stage 2 Random Forest...")
print(f"Training attack flows: {len(X_train_stage2):,}")
print(f"Attack classes: {y_train_stage2.nunique()}")

rf_multiclass_model.fit(
    X_train_stage2,
    y_train_stage2
)

print("\nStage 2 training complete.")

Training Stage 2 Random Forest...
Training attack flows: 297,531
Attack classes: 14

Stage 2 training complete.


In [37]:
stage2_predictions = []

batch_size = 10000

print("Starting Stage 2 validation prediction...")

for start in range(0, len(X_val_stage2), batch_size):
    end = min(start + batch_size, len(X_val_stage2))

    batch_predictions = rf_multiclass_model.predict(
        X_val_stage2.iloc[start:end]
    )

    stage2_predictions.append(batch_predictions)

    print(f"Processed {end:,} / {len(X_val_stage2):,}")

y_val_stage2_pred = np.concatenate(stage2_predictions)

print("\nStage 2 validation prediction complete.")
print(f"Total predictions: {len(y_val_stage2_pred):,}")

Starting Stage 2 validation prediction...
Processed 10,000 / 63,756
Processed 20,000 / 63,756
Processed 30,000 / 63,756
Processed 40,000 / 63,756
Processed 50,000 / 63,756
Processed 60,000 / 63,756
Processed 63,756 / 63,756

Stage 2 validation prediction complete.
Total predictions: 63,756


In [38]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

stage2_accuracy = accuracy_score(
    y_val_stage2,
    y_val_stage2_pred
)

stage2_macro_f1 = f1_score(
    y_val_stage2,
    y_val_stage2_pred,
    average="macro"
)

stage2_weighted_f1 = f1_score(
    y_val_stage2,
    y_val_stage2_pred,
    average="weighted"
)

print("--- Stage 2 Multiclass Validation Results ---")
print(f"Accuracy:    {stage2_accuracy:.4f}")
print(f"Macro F1:    {stage2_macro_f1:.4f}")
print(f"Weighted F1: {stage2_weighted_f1:.4f}")

print("\n--- Classification Report ---")
print(
    classification_report(
        y_val_stage2,
        y_val_stage2_pred,
        zero_division=0
    )
)

--- Stage 2 Multiclass Validation Results ---
Accuracy:    0.9974
Macro F1:    0.8833
Weighted F1: 0.9975

--- Classification Report ---
                          precision    recall  f1-score   support

                     Bot       1.00      1.00      1.00       292
                    DDoS       1.00      1.00      1.00     19202
           DoS GoldenEye       0.99      1.00      1.00      1543
                DoS Hulk       1.00      1.00      1.00     25907
        DoS Slowhttptest       1.00      0.99      1.00       784
           DoS slowloris       1.00      1.00      1.00       807
             FTP-Patator       1.00      1.00      1.00       889
              Heartbleed       1.00      1.00      1.00         2
            Infiltration       0.83      1.00      0.91         5
                PortScan       1.00      1.00      1.00     13520
             SSH-Patator       1.00      1.00      1.00       483
  Web Attack Brute Force       0.75      0.62      0.68       221
Web 

In [39]:
# Analyze Stage 2 Web Attack classification errors

web_attack_labels = [
    "Web Attack Brute Force",
    "Web Attack XSS",
    "Web Attack Sql Injection"
]

web_mask = y_val_stage2.isin(web_attack_labels)

web_results = pd.DataFrame({
    "Actual": y_val_stage2[web_mask].to_numpy(),
    "Predicted": y_val_stage2_pred[web_mask]
})

web_confusion = pd.crosstab(
    web_results["Actual"],
    web_results["Predicted"],
    margins=True
)

print("--- Web Attack Prediction Analysis ---")
print(web_confusion)

--- Web Attack Prediction Analysis ---
Predicted                 DoS GoldenEye  Web Attack Brute Force  \
Actual                                                            
Web Attack Brute Force                0                     136   
Web Attack Sql Injection              2                       0   
Web Attack XSS                        0                      42   
All                                   2                     178   

Predicted                 Web Attack Sql Injection  Web Attack XSS  All  
Actual                                                                   
Web Attack Brute Force                           2              83  221  
Web Attack Sql Injection                         1               0    3  
Web Attack XSS                                   0              56   98  
All                                              3             139  322  


## Stage 2 — Attack Type Classification

After Stage 1 identifies a flow as malicious, Stage 2 classifies the detected attack into one of 14 attack categories.

A class-weighted Random Forest was trained using all 297,531 malicious training flows and evaluated on 63,756 malicious validation flows.

### Validation Results

- Accuracy: 99.74%
- Macro F1: 88.33%
- Weighted F1: 99.75%

The model achieved near-perfect performance on most high-support attack classes, including DDoS, DoS Hulk, PortScan, FTP-Patator, SSH-Patator, and several DoS variants.

### Web Attack Analysis

The main classification difficulty occurred between closely related Web Attack categories.

Of 221 Web Attack Brute Force samples:

- 136 were correctly classified
- 83 were classified as Web Attack XSS
- 2 were classified as Web Attack SQL Injection

Of 98 Web Attack XSS samples:

- 56 were correctly classified
- 42 were classified as Web Attack Brute Force

Web Attack SQL Injection had only 3 validation samples, making its individual performance estimate highly unstable.

### Interpretation

The difference between the Weighted F1 (99.75%) and Macro F1 (88.33%) highlights the strong class imbalance in CIC-IDS2017.

Large attack classes dominate the aggregate performance, while rare Web Attack categories remain more difficult to distinguish.

The results therefore demonstrate strong classification of well-represented known attacks while also exposing limitations for rare and closely related attack classes.

In [40]:
print("--- Final Test Set Check ---")

print(f"Total ML dataset rows: {len(ml_df):,}")
print(f"Test indices:          {len(test_idx):,}")

test_df = ml_df.iloc[test_idx].copy()

print(f"Final test rows:       {len(test_df):,}")
print(f"Test attack classes:   {test_df['Label'].nunique()}")

print("\nTest label distribution:")
print(test_df["Label"].value_counts())

--- Final Test Set Check ---
Total ML dataset rows: 2,519,404
Test indices:          377,911
Final test rows:       377,911
Test attack classes:   15

Test label distribution:
Label
BENIGN                      314154
DoS Hulk                     25908
DDoS                         19201
PortScan                     13519
DoS GoldenEye                 1543
FTP-Patator                    890
DoS slowloris                  808
DoS Slowhttptest               785
SSH-Patator                    483
Bot                            292
Web Attack Brute Force         220
Web Attack XSS                  98
Infiltration                     6
Web Attack Sql Injection         3
Heartbleed                       1
Name: count, dtype: int64


In [41]:
# Stage 1 — Final Test Preparation

X_test = test_df[selected_features]

y_test_binary = (
    test_df["Label"] != "BENIGN"
).astype(int)

print("--- Stage 1 Final Test ---")
print(f"Test flows: {len(X_test):,}")

print("\nBinary distribution:")
print(y_test_binary.value_counts().rename(index={
    0: "BENIGN",
    1: "ATTACK"
}))

--- Stage 1 Final Test ---
Test flows: 377,911

Binary distribution:
Label
BENIGN    314154
ATTACK     63757
Name: count, dtype: int64


In [42]:
# Stage 1 — Final Test Prediction

test_predictions = []
batch_size = 10000

print("Starting Stage 1 final test prediction...")

for start in range(0, len(X_test), batch_size):
    end = min(start + batch_size, len(X_test))

    batch_predictions = rf_full_model.predict(
        X_test.iloc[start:end]
    )

    test_predictions.append(batch_predictions)

    if end % 50000 == 0 or end == len(X_test):
        print(f"Processed {end:,} / {len(X_test):,}")

y_test_pred = np.concatenate(test_predictions)

print("\nStage 1 final test prediction complete.")
print(f"Total predictions: {len(y_test_pred):,}")

Starting Stage 1 final test prediction...
Processed 50,000 / 377,911
Processed 100,000 / 377,911
Processed 150,000 / 377,911
Processed 200,000 / 377,911
Processed 250,000 / 377,911
Processed 300,000 / 377,911
Processed 350,000 / 377,911
Processed 377,911 / 377,911

Stage 1 final test prediction complete.
Total predictions: 377,911


In [43]:
# Stage 1 — Final Test Metrics

test_accuracy = accuracy_score(
    y_test_binary,
    y_test_pred
)

test_precision = precision_score(
    y_test_binary,
    y_test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test_binary,
    y_test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test_binary,
    y_test_pred,
    zero_division=0
)

test_cm = confusion_matrix(
    y_test_binary,
    y_test_pred
)

test_tn, test_fp, test_fn, test_tp = test_cm.ravel()

print("--- Stage 1 Final Test Results ---")
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1 Score:  {test_f1:.4f}")

print("\nConfusion Matrix:")
print(test_cm)

print("\n--- Detection Summary ---")
print(f"True Negatives:  {test_tn:,}")
print(f"False Positives: {test_fp:,}")
print(f"False Negatives: {test_fn:,}")
print(f"True Positives:  {test_tp:,}")

--- Stage 1 Final Test Results ---
Accuracy:  0.9992
Precision: 0.9961
Recall:    0.9990
F1 Score:  0.9975

Confusion Matrix:
[[313904    250]
 [    63  63694]]

--- Detection Summary ---
True Negatives:  313,904
False Positives: 250
False Negatives: 63
True Positives:  63,694


In [44]:
# Stage 2 — Final Test Preparation

test_stage2 = test_df[
    test_df["Label"] != "BENIGN"
].copy()

X_test_stage2 = test_stage2[selected_features]
y_test_stage2 = test_stage2["Label"]

print("--- Stage 2 Final Test ---")
print(f"Attack flows:   {len(X_test_stage2):,}")
print(f"Attack classes: {y_test_stage2.nunique()}")

print("\nAttack distribution:")
print(y_test_stage2.value_counts())

--- Stage 2 Final Test ---
Attack flows:   63,757
Attack classes: 14

Attack distribution:
Label
DoS Hulk                    25908
DDoS                        19201
PortScan                    13519
DoS GoldenEye                1543
FTP-Patator                   890
DoS slowloris                 808
DoS Slowhttptest              785
SSH-Patator                   483
Bot                           292
Web Attack Brute Force        220
Web Attack XSS                 98
Infiltration                    6
Web Attack Sql Injection        3
Heartbleed                      1
Name: count, dtype: int64


In [45]:
# Stage 2 — Final Test Prediction

stage2_test_predictions = []
batch_size = 10000

print("Starting Stage 2 final test prediction...")

for start in range(0, len(X_test_stage2), batch_size):
    end = min(start + batch_size, len(X_test_stage2))

    batch_predictions = rf_multiclass_model.predict(
        X_test_stage2.iloc[start:end]
    )

    stage2_test_predictions.append(batch_predictions)

    print(f"Processed {end:,} / {len(X_test_stage2):,}")

y_test_stage2_pred = np.concatenate(stage2_test_predictions)

print("\nStage 2 final test prediction complete.")
print(f"Total predictions: {len(y_test_stage2_pred):,}")

Starting Stage 2 final test prediction...
Processed 10,000 / 63,757
Processed 20,000 / 63,757
Processed 30,000 / 63,757
Processed 40,000 / 63,757
Processed 50,000 / 63,757
Processed 60,000 / 63,757
Processed 63,757 / 63,757

Stage 2 final test prediction complete.
Total predictions: 63,757


In [46]:
# Stage 2 — Final Test Metrics

stage2_test_accuracy = accuracy_score(
    y_test_stage2,
    y_test_stage2_pred
)

stage2_test_macro_f1 = f1_score(
    y_test_stage2,
    y_test_stage2_pred,
    average="macro"
)

stage2_test_weighted_f1 = f1_score(
    y_test_stage2,
    y_test_stage2_pred,
    average="weighted"
)

print("--- Stage 2 Final Test Results ---")
print(f"Accuracy:    {stage2_test_accuracy:.4f}")
print(f"Macro F1:    {stage2_test_macro_f1:.4f}")
print(f"Weighted F1: {stage2_test_weighted_f1:.4f}")

print("\n--- Final Classification Report ---")
print(
    classification_report(
        y_test_stage2,
        y_test_stage2_pred,
        zero_division=0
    )
)

--- Stage 2 Final Test Results ---
Accuracy:    0.9973
Macro F1:    0.9073
Weighted F1: 0.9975

--- Final Classification Report ---
                          precision    recall  f1-score   support

                     Bot       1.00      1.00      1.00       292
                    DDoS       1.00      1.00      1.00     19201
           DoS GoldenEye       0.99      1.00      1.00      1543
                DoS Hulk       1.00      1.00      1.00     25908
        DoS Slowhttptest       0.99      0.99      0.99       785
           DoS slowloris       0.99      0.99      0.99       808
             FTP-Patator       1.00      1.00      1.00       890
              Heartbleed       1.00      1.00      1.00         1
            Infiltration       0.67      1.00      0.80         6
                PortScan       1.00      1.00      1.00     13519
             SSH-Patator       1.00      1.00      1.00       483
  Web Attack Brute Force       0.77      0.58      0.66       220
Web Attac

# Final Model Evaluation

After model selection was completed using the training and validation sets, the untouched test set was used once for final evaluation.

The final test set contains 377,911 network flows.

## Stage 1 — Binary Intrusion Detection

The Stage 1 Random Forest determines whether a network flow is BENIGN or ATTACK.

### Final Test Results

- Accuracy: 99.92%
- Precision: 99.61%
- Recall: 99.90%
- F1 Score: 99.75%

Confusion matrix:

- True Negatives: 313,904
- False Positives: 250
- False Negatives: 63
- True Positives: 63,694

Out of 63,757 malicious flows in the final test set, the detector correctly identified 63,694 and missed 63.

The final test performance closely matches the validation results, indicating stable performance on the stratified random split.

However, the separate source-holdout stress test showed that this strong performance should not be interpreted as reliable detection of completely unseen attack patterns.

---

## Stage 2 — Attack Type Classification

Flows identified as malicious are passed to Stage 2, where a second Random Forest classifies them into one of 14 attack categories.

The final Stage 2 test contains 63,757 malicious flows.

### Final Test Results

- Accuracy: 99.73%
- Macro F1: 90.73%
- Weighted F1: 99.75%

The classifier achieved near-perfect performance on most well-represented attack categories, including DDoS, DoS Hulk, PortScan, FTP-Patator, SSH-Patator, and several DoS variants.

The main classification difficulty remained within the Web Attack family:

- Web Attack Brute Force: F1 = 0.66
- Web Attack XSS: F1 = 0.46
- Web Attack SQL Injection: F1 = 0.80 (3 test samples)

Rare-class results must be interpreted cautiously:

- Heartbleed: 1 test sample
- SQL Injection: 3 test samples
- Infiltration: 6 test samples

These sample sizes are too small to support strong conclusions about real-world performance for those individual classes.

## Final Architecture

Network Flow
→ Stage 1: BENIGN / ATTACK
→ Stage 2 (if ATTACK): Attack Type
→ Alert / Dashboard

Overall, the supervised pipeline performs strongly on attack patterns represented during training, while the source-holdout experiment demonstrates an important limitation when encountering previously unseen attack patterns.

In [47]:
from pathlib import Path
import joblib

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

print(f"Models directory: {MODELS_DIR.resolve()}")
print(f"joblib version: {joblib.__version__}")

Models directory: C:\Users\abdal\SentinelFlow-AI\models
joblib version: 1.6.0


In [48]:
# Save final SentinelFlow AI models and feature list

stage1_path = MODELS_DIR / "stage1_binary_random_forest.joblib"
stage2_path = MODELS_DIR / "stage2_attack_classifier_random_forest.joblib"
features_path = MODELS_DIR / "selected_features.joblib"

print("Saving SentinelFlow AI models...")

joblib.dump(rf_full_model, stage1_path)
print(f"Stage 1 saved: {stage1_path}")

joblib.dump(rf_multiclass_model, stage2_path)
print(f"Stage 2 saved: {stage2_path}")

joblib.dump(selected_features, features_path)
print(f"Feature list saved: {features_path}")

print("\n--- Saved Artifacts ---")
print(f"Stage 1 model: {stage1_path.stat().st_size / (1024**2):.2f} MB")
print(f"Stage 2 model: {stage2_path.stat().st_size / (1024**2):.2f} MB")
print(f"Selected features: {len(selected_features)}")

print("\nSentinelFlow AI model artifacts saved successfully.")

Saving SentinelFlow AI models...
Stage 1 saved: models\stage1_binary_random_forest.joblib
Stage 2 saved: models\stage2_attack_classifier_random_forest.joblib
Feature list saved: models\selected_features.joblib

--- Saved Artifacts ---
Stage 1 model: 22.23 MB
Stage 2 model: 17.74 MB
Selected features: 65

SentinelFlow AI model artifacts saved successfully.


In [49]:
# Verify saved model artifacts

loaded_stage1 = joblib.load(
    MODELS_DIR / "stage1_binary_random_forest.joblib"
)

loaded_stage2 = joblib.load(
    MODELS_DIR / "stage2_attack_classifier_random_forest.joblib"
)

loaded_features = joblib.load(
    MODELS_DIR / "selected_features.joblib"
)

print("--- Artifact Verification ---")
print(f"Stage 1 loaded: {type(loaded_stage1).__name__}")
print(f"Stage 2 loaded: {type(loaded_stage2).__name__}")
print(f"Features loaded: {len(loaded_features)}")

print(
    "Feature list matches:",
    loaded_features == selected_features
)

# Test one BENIGN flow
benign_sample = test_df[
    test_df["Label"] == "BENIGN"
].iloc[[0]]

benign_prediction = loaded_stage1.predict(
    benign_sample[loaded_features]
)[0]

# Test one ATTACK flow
attack_sample = test_df[
    test_df["Label"] != "BENIGN"
].iloc[[0]]

attack_binary_prediction = loaded_stage1.predict(
    attack_sample[loaded_features]
)[0]

attack_type_prediction = loaded_stage2.predict(
    attack_sample[loaded_features]
)[0]

print("\n--- Prediction Sanity Check ---")
print(f"BENIGN sample -> Stage 1 prediction: {benign_prediction}")
print(f"ATTACK sample actual label:          {attack_sample['Label'].iloc[0]}")
print(f"ATTACK sample -> Stage 1 prediction: {attack_binary_prediction}")
print(f"ATTACK sample -> Stage 2 prediction: {attack_type_prediction}")

--- Artifact Verification ---
Stage 1 loaded: RandomForestClassifier
Stage 2 loaded: RandomForestClassifier
Features loaded: 65
Feature list matches: True

--- Prediction Sanity Check ---
BENIGN sample -> Stage 1 prediction: 0
ATTACK sample actual label:          DoS Hulk
ATTACK sample -> Stage 1 prediction: 1
ATTACK sample -> Stage 2 prediction: DoS Hulk


In [50]:
# Save SentinelFlow model metadata

model_metadata = {
    "project": "SentinelFlow AI",
    "dataset": "CIC-IDS2017",

    "stage1": {
        "task": "Binary Intrusion Detection",
        "model": "RandomForestClassifier",
        "labels": {
            0: "BENIGN",
            1: "ATTACK"
        }
    },

    "stage2": {
        "task": "Attack Type Classification",
        "model": "RandomForestClassifier",
        "classes": list(loaded_stage2.classes_)
    },

    "feature_count": len(loaded_features)
}

metadata_path = MODELS_DIR / "model_metadata.joblib"

joblib.dump(model_metadata, metadata_path)

print("--- Model Metadata Saved ---")
print(f"Path: {metadata_path}")
print(f"Feature count: {model_metadata['feature_count']}")
print(f"Stage 1 labels: {model_metadata['stage1']['labels']}")
print(f"Stage 2 classes: {len(model_metadata['stage2']['classes'])}")

print("\nAttack classes:")
for attack_class in model_metadata["stage2"]["classes"]:
    print(f"- {attack_class}")

--- Model Metadata Saved ---
Path: models\model_metadata.joblib
Feature count: 65
Stage 1 labels: {0: 'BENIGN', 1: 'ATTACK'}
Stage 2 classes: 14

Attack classes:
- Bot
- DDoS
- DoS GoldenEye
- DoS Hulk
- DoS Slowhttptest
- DoS slowloris
- FTP-Patator
- Heartbleed
- Infiltration
- PortScan
- SSH-Patator
- Web Attack Brute Force
- Web Attack Sql Injection
- Web Attack XSS
